# RPMT Cross-Modal Feature-Space Analysis

This notebook builds publication-ready PCA and t-SNE evidence from **real positive relations**. It keeps three claims separate: the geometry of visual relations, the discriminability of debiased triplet anchors, and cross-modal correspondence after projection. Ground-truth predicates are used only for sampling, coloring, and analysis; they are never passed to the model during inference.

> Checkpoints in this legacy stack are loaded with `torch.load`; use only trusted files. A single RPMT checkpoint supports the three-stage analysis. A genuine causal before/after comparison additionally requires a checkpoint trained without visual structure preservation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'visualization':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from visualization.rpmt_feature_analysis import (
    collect_checkpoint_features, export_figure, load_analysis_runtime,
    load_feature_cache, make_ablation_figure, make_anchor_refinement_figure,
    make_split_comparison_figure, make_three_stage_figure,
    sample_dataset_occurrences, save_feature_cache, summarize_metrics,
)

## 1. Configuration

Only the model YAML and full checkpoint are required. `SELECTED_PREDICATES` deliberately has no automatic fallback: inspect the support table first, then choose predicates based on semantic relevance and adequate support rather than visual appearance.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs/e2e_relation_X_101_32_8_FPN_1x_total.yaml'
CHECKPOINT_PATH = Path('/absolute/path/to/rpmt_checkpoint.pth')
ABLATION_CHECKPOINT_PATH = None  # Optional: checkpoint trained without visual structure preservation
DATASET_NAME = 'VG'              # Use the corresponding YAML to switch to GQA
SELECTED_PREDICATES = []         # Fill after inspecting the support table below
MAX_SAMPLES_PER_CLASS = 80
RANDOM_SEED = 2027
TSNE_PERPLEXITY = 30.0
KNN_K = 10
DEVICE = None                    # None selects CUDA when available
OUTPUT_DIR = PROJECT_ROOT / 'visualization/outputs/rpmt_feature_space'
CACHE_DIR = PROJECT_ROOT / 'visualization/cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the model and inspect predicate support

The notebook forces the test dataset to the total predicate split and uses one image per batch so projector calls remain exactly aligned with relation pairs. This changes neither checkpoint weights nor extracted representations.

In [ ]:
runtime = load_analysis_runtime(
    CONFIG_PATH, CHECKPOINT_PATH, dataset_name=DATASET_NAME, device=DEVICE
)
pd.Series(runtime['compatibility'], name='value').to_frame()

In [ ]:
support, _ = sample_dataset_occurrences(
    runtime['dataset'], [], max_per_class=MAX_SAMPLES_PER_CLASS, seed=RANDOM_SEED
)
predictor = runtime['predictor']
seen_ids = set(int(i) for i in predictor.base)
unseen_ids = set(int(i) for i in predictor.novel)
name_to_id = {name: i for i, name in enumerate(runtime['dataset'].ind_to_predicates)}
support_table = pd.DataFrame([
    {
        'predicate': name,
        'split': 'seen' if name_to_id[name] in seen_ids else ('unseen' if name_to_id[name] in unseen_ids else 'other'),
        'positive_instances': count,
    }
    for name, count in support.items()
]).sort_values(['split', 'positive_instances'], ascending=[True, False], ignore_index=True)
support_table

### Predicate-selection rule

Choose a small, fixed set containing both seen and unseen predicates. Prefer predicates that express meaningfully related or confusable interactions and have enough test support. Record the chosen list before examining PCA/t-SNE layouts. The plotting functions reject an empty list by design.

In [ ]:
# Example syntax only; replace with the predicates selected from support_table.
# SELECTED_PREDICATES = ['on', 'standing on', 'holding', 'carrying', ...]
assert SELECTED_PREDICATES, (
    'Set SELECTED_PREDICATES after inspecting support_table; no classes are selected automatically.'
)

## 3. Extract and cache paired features

For every selected GT relation, the cache stores $\mathbf v_i$ (the visual representation used by the structure loss), $\mathbf z_i=f(\mathbf v_i)$, the raw CLIP triplet anchor $\mathbf a_i$, and its SVD-refined version $\mathbf t_i$. Sampling is balanced per predicate with a fixed seed.

In [ ]:
support, sampled_occurrences = sample_dataset_occurrences(
    runtime['dataset'], SELECTED_PREDICATES,
    max_per_class=MAX_SAMPLES_PER_CLASS, seed=RANDOM_SEED,
)
cache_path = CACHE_DIR / f'{DATASET_NAME.lower()}_{CHECKPOINT_PATH.stem}_seed{RANDOM_SEED}.pt'
if cache_path.exists():
    arrays, cache_metadata = load_feature_cache(cache_path)
    expected_cache_fields = {
        'selected_predicates': list(SELECTED_PREDICATES),
        'checkpoint': str(CHECKPOINT_PATH), 'config': str(CONFIG_PATH),
        'max_samples_per_class': MAX_SAMPLES_PER_CLASS, 'seed': RANDOM_SEED,
    }
    mismatched = [key for key, value in expected_cache_fields.items() if cache_metadata.get(key) != value]
    if mismatched:
        raise ValueError(f'Existing cache metadata differs for {mismatched}; remove or rename it.')
else:
    arrays = collect_checkpoint_features(runtime, sampled_occurrences)
    cache_metadata = {
        'dataset': DATASET_NAME, 'config': str(CONFIG_PATH),
        'checkpoint': str(CHECKPOINT_PATH), 'selected_predicates': list(SELECTED_PREDICATES),
        'max_samples_per_class': MAX_SAMPLES_PER_CLASS, 'seed': RANDOM_SEED,
        'gt_labels_used_only_for_analysis': True,
    }
    save_feature_cache(cache_path, arrays, cache_metadata)
print(f'Loaded {len(arrays["predicate_id"])} real positive relations from {cache_path}')

## 4. High-dimensional quantitative evidence

These values, rather than distances in the 2-D plots, support the claims. Lower structure MAE, higher Spearman correlation and neighborhood overlap indicate better preservation of visual relation geometry; higher alignment cosine indicates stronger correspondence with the refined semantic anchors.

In [ ]:
metric_table = pd.DataFrame(summarize_metrics(arrays, k=KNN_K))
metric_table.round(4)

## 5. Export PCA and t-SNE candidates

Circles denote real visual relation instances and hollow diamonds denote their triplet semantic anchors. Every shared-space panel is produced by one joint reducer fit. PDF is intended for LaTeX; PNG is exported at 600 dpi for inspection.

In [ ]:
generated = []
for reducer in ('pca', 'tsne'):
    kwargs = dict(method=reducer, seed=RANDOM_SEED, perplexity=TSNE_PERPLEXITY)
    figures = {
        'three_stage': make_three_stage_figure(arrays, SELECTED_PREDICATES, **kwargs),
        'split_2x2': make_split_comparison_figure(arrays, SELECTED_PREDICATES, compact=False, **kwargs),
        'compact_1x2': make_split_comparison_figure(arrays, SELECTED_PREDICATES, compact=True, **kwargs),
        'anchor_refinement': make_anchor_refinement_figure(arrays, SELECTED_PREDICATES, **kwargs),
    }
    for figure_name, figure in figures.items():
        generated.extend(export_figure(figure, OUTPUT_DIR / f'{DATASET_NAME.lower()}_{reducer}_{figure_name}'))
        plt.show()
        plt.close(figure)
generated

## 6. Optional causal ablation comparison

Run this section only with a checkpoint trained without visual structure preservation. It evaluates exactly the same sampled image relations and jointly reduces both projected spaces with the same semantic anchors.

In [ ]:
if ABLATION_CHECKPOINT_PATH is not None:
    ablation_runtime = load_analysis_runtime(
        CONFIG_PATH, ABLATION_CHECKPOINT_PATH, dataset_name=DATASET_NAME, device=DEVICE
    )
    ablation_arrays = collect_checkpoint_features(ablation_runtime, sampled_occurrences)
    display(pd.concat({
        'w/o structure': pd.DataFrame(summarize_metrics(ablation_arrays, k=KNN_K)),
        'RPMT': pd.DataFrame(summarize_metrics(arrays, k=KNN_K)),
    }, names=['model']))
    for reducer in ('pca', 'tsne'):
        figure = make_ablation_figure(
            arrays, ablation_arrays, SELECTED_PREDICATES,
            method=reducer, seed=RANDOM_SEED, perplexity=TSNE_PERPLEXITY,
        )
        export_figure(figure, OUTPUT_DIR / f'{DATASET_NAME.lower()}_{reducer}_ablation')
        plt.show()
        plt.close(figure)
else:
    print('No ablation checkpoint supplied; skipping causal before/after comparison.')

## Interpretation checklist

- Select predicates before inspecting the embeddings and report the selection rule.
- Do not interpret global orientation or absolute distances across separately fitted panels.
- Use PCA as the primary reproducible view; treat t-SNE as a qualitative alternative.
- Support structure-preservation claims with the high-dimensional metrics above.
- Use `before/after` wording only when the optional no-structure checkpoint is supplied.
- Keep synthesized pseudo-novel features out of this Introduction figure.